In [1]:
import pyvisa
import numpy as np

In [2]:
#Connecting to awg
rm = pyvisa.ResourceManager()
my_instrument = rm.open_resource('TCPIP::192.168.50.65::INSTR')


In [3]:
print(my_instrument.query('*IDN?'))

TEKTRONIX,AWG7122C,B051135,SCPI:99.0 FW:4.6.0.7 



In [73]:
#Uploading a sawtooth wave,uysing float (DOES NOT WORK< NEED TO APPEND HTE MARKER TO THE FLOAT TO GET THIS)
sampl_rate = 12 #in GSa/s
freq = 1 #in GHz
ampl = 0.5 #in V

num_samples = int(np.round(sampl_rate/freq))
voltages = np.linspace(0, ampl, int(num_samples))
print(voltages)
print(voltages.shape)
#voltages = [1,0,1,0,1,0,0,1,0,1,0,1]


#Create new waveform
my_instrument.write(f'WLIS:WAV:DEL "sawtooth"')
my_instrument.write(f'WLISt:WAVeform:NEW "sawtooth",{num_samples},REAL')
my_instrument.write_binary_values(f'WLISt:WAVeform:DATA "sawtooth",0,{num_samples},', voltages.tolist(), datatype='f', header_fmt='ieee')

[0.         0.04545455 0.09090909 0.13636364 0.18181818 0.22727273
 0.27272727 0.31818182 0.36363636 0.40909091 0.45454545 0.5       ]
(12,)


(90, <StatusCode.success: 0>)

In [9]:
#Uploading a sawtooth wave, iinteger
sampl_rate = 12 #in GSa/s
freq = 1  #in GHz
ampl = 1

if (ampl < 0.5):
    ref_ampl = 0.5
else:
    ref_ampl = 1

#Turn off output
my_instrument.write('OUTP1 OFF')
#First set awg parameters, scaling the amplitude to correct values
my_instrument.write(f'SOURCE1:VOLTAGE:AMPLITUDE {ref_ampl}')
#Also set to use 10bit DAC mode
my_instrument.write(f'SOURCE1:DAC:RES 10')

num_samples = int(np.round(sampl_rate/freq))
AWG_MAX_INT = 2**10-1 #Assuming 10 bit DAQ mode

waveform_scale_int = (AWG_MAX_INT)*ampl/ref_ampl


BIT_POS = 4 #where lowest bit is in the byte for sending
voltages = (np.round(np.linspace((AWG_MAX_INT)/2-waveform_scale_int/2, (AWG_MAX_INT)/2+waveform_scale_int/2, int(num_samples)))).astype(int)
print(voltages)

tosend = (np.left_shift(voltages, BIT_POS))
my_instrument.write(f'WLIS:WAV:DEL "sawtooth"')
my_instrument.write(f'WLISt:WAVeform:NEW "sawtooth",{num_samples},INT')
my_instrument.write_binary_values(f'WLISt:WAVeform:DATA "sawtooth",0,{num_samples},', tosend.tolist(), datatype='H', header_fmt='ieee')

#Finally set output to use the desired wave 
my_instrument.write(f'SOURCE1:WAV "sawtooth"')

#Return on the output
my_instrument.write('OUTP1 ON')

my_instrument.write('AWGC:RUN')

[   0   93  186  279  372  465  558  651  744  837  930 1023]


(10, <StatusCode.success: 0>)